In [0]:
CATALOG = "REPLACE_CATALOG"
SCHEMA = "northstar4_REPLACE_LEARNER"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`stream_state`
""")

NS = f"`{CATALOG}`.`{SCHEMA}`"

STREAM_BRONZE = f"{NS}.`sales_stream_bronze`"
STREAM_GOLD = f"{NS}.`sales_stream_gold`"

STATE = f"/Volumes/{CATALOG}/{SCHEMA}/stream_state"
BRONZE_CP = f"{STATE}/sales_demo/bronze_v1"
GOLD_CP = f"{STATE}/sales_demo/gold_v1"

print("Namespace:", NS)
print("Bronze:", STREAM_BRONZE)
print("Gold:", STREAM_GOLD)
print("Bronze checkpoint:", BRONZE_CP)
print("Gold checkpoint:", GOLD_CP)

Namespace: `cziane_db`.`northstar4_demo_cziane`
Bronze: `cziane_db`.`northstar4_demo_cziane`.`sales_stream_bronze`
Gold: `cziane_db`.`northstar4_demo_cziane`.`sales_stream_gold`
Bronze checkpoint: /Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1
Gold checkpoint: /Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/gold_v1


In [0]:
EH_NAMESPACE = "REPLACE_NAMESPACE.servicebus.windows.net"
EH_NAME = "sales-demo"
SERVICE_CREDENTIAL = "REPLACE_SERVICE_CREDENTIAL"

kafka_options = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}:9093",
    "subscribe": EH_NAME,
    "databricks.serviceCredential": SERVICE_CREDENTIAL,
    "startingOffsets": "earliest",
    "failOnDataLoss": "true",
    "maxOffsetsPerTrigger": "1000",
}

kafka_stream = (
    spark.readStream
        .format("kafka")
        .options(**kafka_options)
        .load()
)

print("isStreaming:", kafka_stream.isStreaming)
kafka_stream.printSchema()

isStreaming: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

event_schema = """
    sale_id STRING,
    store_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DECIMAL(10,2),
    event_time TIMESTAMP
"""

incoming = (
    kafka_stream
    .select(
        F.col("value").cast("string").alias("raw_json"),
        "topic",
        "partition",
        "offset",
        F.col("timestamp").alias("broker_time")
    )
    .withColumn(
        "event",
        F.from_json(
            "raw_json",
            event_schema,
            {"mode": "PERMISSIVE"}
        )
    )
    .select(
        "raw_json",
        "topic",
        "partition",
        "offset",
        "broker_time",
        "event.*"
    )
    .withColumn(
        "processing_time",
        F.current_timestamp()
    )
)

incoming.printSchema()

root
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- broker_time: timestamp (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- processing_time: timestamp (nullable = false)



In [0]:
BRONZE_CP = f"{STATE}/sales_demo/bronze_v1"

bronze_query = (
    incoming.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CP)
    .queryName("northstar4_bronze")
    .trigger(availableNow=True)
    .toTable(STREAM_BRONZE)
)

bronze_query.awaitTermination()

print("Bronze query finished")
print("Active:", bronze_query.isActive)

Bronze query finished
Active: False


In [0]:
bronze = spark.table(STREAM_BRONZE)

print("Bronze rows:", bronze.count())

bronze.select(
    "sale_id",
    "store_id",
    "product_id",
    "quantity",
    "unit_price",
    "event_time",
    "broker_time",
    "processing_time",
    "partition",
    "offset"
).orderBy("sale_id").show(30, truncate=False)

bronze.select("sale_id").distinct().count()

Bronze rows: 8
+-----------+--------+----------+--------+----------+-------------------+-----------------------+-----------------------+---------+------+
|sale_id    |store_id|product_id|quantity|unit_price|event_time         |broker_time            |processing_time        |partition|offset|
+-----------+--------+----------+--------+----------+-------------------+-----------------------+-----------------------+---------+------+
|DEMO-000001|S01     |P01       |1       |4.50      |2026-10-01 08:00:00|2026-10-01 09:27:04.939|2026-10-01 10:08:35.882|1        |0     |
|DEMO-000002|S02     |P02       |2       |12.00     |2026-10-01 08:00:05|2026-10-01 09:27:05.455|2026-10-01 10:08:35.882|1        |1     |
|DEMO-000003|S03     |P03       |3       |7.25      |2026-10-01 08:00:10|2026-10-01 09:27:05.981|2026-10-01 10:08:35.882|0        |0     |
|DEMO-000004|S04     |P04       |1       |19.90     |2026-10-01 08:00:15|2026-10-01 09:27:06.121|2026-10-01 10:08:35.882|0        |1     |
|DEMO-000005

8

In [0]:
display(dbutils.fs.ls(BRONZE_CP))

path,name,size,modificationTime
dbfs:/Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1/__tmp_path_dir/,__tmp_path_dir/,0,1790849310000
dbfs:/Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1/commits/,commits/,0,1790849310000
dbfs:/Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1/metadata,metadata,45,1790849311000
dbfs:/Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1/offsets/,offsets/,0,1790849310000
dbfs:/Volumes/cziane_db/northstar4_demo_cziane/stream_state/sales_demo/bronze_v1/sources/,sources/,0,1790849311000


In [0]:
from pyspark.sql import functions as F

bronze_stream = spark.readStream.table(STREAM_BRONZE)

valid_events = bronze_stream.filter(
    F.col("sale_id").isNotNull()
    & (F.trim("sale_id") != "")
    & F.col("store_id").isNotNull()
    & (F.trim("store_id") != "")
    & F.col("product_id").isNotNull()
    & F.col("event_time").isNotNull()
    & (F.col("quantity") > 0)
    & (F.col("unit_price") > 0)
)

with_revenue = valid_events.withColumn(
    "revenue",
    (F.col("quantity") * F.col("unit_price")).cast("decimal(18,2)")
)

windows = (
    with_revenue
    .groupBy(
        "store_id",
        F.window("event_time", "1 minute")
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").cast("decimal(20,2)").alias("total_revenue")
    )
)

stream_gold = windows.select(
    "store_id",
    F.col("window.start").alias("window_start"),
    F.col("window.end").alias("window_end"),
    "transaction_count",
    "total_quantity",
    "total_revenue"
)

stream_gold.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)



In [0]:
GOLD_CP = f"{STATE}/sales_demo/gold_v1"

gold_query = (
    stream_gold.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", GOLD_CP)
    .queryName("northstar4_gold")
    .trigger(availableNow=True)
    .toTable(STREAM_GOLD)
)

gold_query.awaitTermination()

print("Gold query finished")
print("Active:", gold_query.isActive)

Gold query finished
Active: False


In [0]:
gold = spark.table(STREAM_GOLD)

gold.orderBy("window_start", "store_id").show(30, truncate=False)

+--------+-------------------+-------------------+-----------------+--------------+-------------+
|store_id|window_start       |window_end         |transaction_count|total_quantity|total_revenue|
+--------+-------------------+-------------------+-----------------+--------------+-------------+
|S01     |2026-10-01 08:00:00|2026-10-01 08:01:00|2                |3             |12.00        |
|S02     |2026-10-01 08:00:00|2026-10-01 08:01:00|2                |5             |49.20        |
|S03     |2026-10-01 08:00:00|2026-10-01 08:01:00|2                |4             |26.25        |
|S04     |2026-10-01 08:00:00|2026-10-01 08:01:00|2                |3             |43.90        |
+--------+-------------------+-------------------+-----------------+--------------+-------------+



In [0]:
from decimal import Decimal

totals = (
    gold.agg(
        F.sum("transaction_count").alias("transactions"),
        F.sum("total_quantity").alias("quantity"),
        F.sum("total_revenue").alias("revenue")
    )
    .first()
)

print("Gold rows:", gold.count())
print("Transactions:", totals["transactions"])
print("Quantity:", totals["quantity"])
print("Revenue:", totals["revenue"])

Gold rows: 4
Transactions: 8
Quantity: 15
Revenue: 131.35


In [0]:
before_bronze = spark.table(STREAM_BRONZE).count()

before_gold = (
    spark.table(STREAM_GOLD)
    .orderBy("store_id", "window_start")
    .collect()
)

print("Before restart:")
print("Bronze rows:", before_bronze)
print("Gold rows:", len(before_gold))

Before restart:
Bronze rows: 8
Gold rows: 4


In [0]:
replay_bronze = (
    incoming.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CP)
    .trigger(availableNow=True)
    .toTable(STREAM_BRONZE)
)

replay_bronze.awaitTermination()

print("Bronze replay finished")
print("Active:", replay_bronze.isActive)

Bronze replay finished
Active: False


In [0]:
replay_gold = (
    stream_gold.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", GOLD_CP)
    .trigger(availableNow=True)
    .toTable(STREAM_GOLD)
)

replay_gold.awaitTermination()

print("Gold replay finished")
print("Active:", replay_gold.isActive)

Gold replay finished
Active: False


In [0]:
after_bronze = spark.table(STREAM_BRONZE).count()

after_gold = (
    spark.table(STREAM_GOLD)
    .orderBy("store_id", "window_start")
    .collect()
)

assert after_bronze == before_bronze
assert after_gold == before_gold

print("PASS checkpoint recovery")
print("Bronze rows:", after_bronze)
print("Gold rows:", len(after_gold))

PASS checkpoint recovery
Bronze rows: 8
Gold rows: 4
